In [2]:
# %pip install openmeteo-requests
# %pip install requests-cache retry-requests
# %pip install polars
# %pip install duckdb
# %pip install Numpy

In [3]:
import openmeteo_requests
import polars as pl
import requests_cache
from retry_requests import retry
from datetime import datetime, timedelta, timezone
import duckdb
import pyarrow

In [4]:
# api config
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

url = "https://api.open-meteo.com/v1/forecast"

Functionise the below to make easier

In [5]:
# Call config
# Oxford
_latitude = 52.52
_longitude = 13.41

# The order of variables in hourly or daily is important to assign them correctly below
# Would be better to define a dictionary of the variables then pass in the names/order to use in later arguments. Then there is no risk of assigning values of one and mislabelling them.
var_list = [
    "temperature_2m", 
    "relative_humidity_2m", 
    "apparent_temperature", 
    "precipitation_probability", 
    "precipitation", "rain", 
    "showers", "snowfall", 
    "snow_depth", 
    "weather_code", 
    "cloud_cover", 
    "wind_speed_10m"
]

params = {
	"latitude": _latitude,
	"longitude": _longitude,
	"hourly": var_list,
}



In [6]:
# api call
responses = openmeteo.weather_api(url, params=params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
hourly = response.Hourly()

In [7]:
# Setting up the datetime length to create the dataframe with
start = datetime.fromtimestamp(hourly.Time(), timezone.utc)
end = datetime.fromtimestamp(hourly.TimeEnd(), timezone.utc)
freq = timedelta(seconds = hourly.Interval())

# Dataframe base
# get the columns before populating with metrics
df_base = pl.select(
    lat = response.Latitude(),
    long = response.Longitude(),
    extract_date = datetime.fromtimestamp(hourly.Time(), timezone.utc).date(),
    extract_time = datetime.now().time(),
    fc_datetime = pl.datetime_range(start, end, freq, closed="left")
).with_columns(
    fc_date = pl.col('fc_datetime').dt.date(),
    fc_hour = pl.col('fc_datetime').dt.strftime("%H")
)

col_exprs = {var: hourly.Variables(var_list.index(var)).ValuesAsNumpy() for var in var_list}

df_metrics = pl.DataFrame(
    col_exprs
)

df_raw = pl.concat([df_base, df_metrics], how='horizontal')

To do:
- Motherduck staging table
- Motherduck main table - recency rank
- Motherduck light table for latest values only (where recency rank = 1)

In [8]:
df_raw

lat,long,extract_date,extract_time,fc_datetime,fc_date,fc_hour,temperature_2m,relative_humidity_2m,apparent_temperature,precipitation_probability,precipitation,rain,showers,snowfall,snow_depth,weather_code,cloud_cover,wind_speed_10m
f64,f64,date,time,"datetime[μs, UTC]",date,str,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
52.52,13.419998,2026-03-21,21:51:07.412808,2026-03-21 00:00:00 UTC,2026-03-21,"""00""",5.8955,94.0,3.291759,0.0,0.0,0.0,0.0,0.0,0.0,3.0,100.0,9.511088
52.52,13.419998,2026-03-21,21:51:07.412808,2026-03-21 01:00:00 UTC,2026-03-21,"""01""",5.8955,93.0,3.681322,0.0,0.0,0.0,0.0,0.0,0.0,3.0,95.0,6.618519
52.52,13.419998,2026-03-21,21:51:07.412808,2026-03-21 02:00:00 UTC,2026-03-21,"""02""",5.9955,93.0,3.611155,0.0,0.0,0.0,0.0,0.0,0.0,3.0,100.0,7.928177
52.52,13.419998,2026-03-21,21:51:07.412808,2026-03-21 03:00:00 UTC,2026-03-21,"""03""",6.2955,92.0,3.900229,0.0,0.0,0.0,0.0,0.0,0.0,3.0,100.0,8.209263
52.52,13.419998,2026-03-21,21:51:07.412808,2026-03-21 04:00:00 UTC,2026-03-21,"""04""",6.3455,91.0,3.916024,0.0,0.0,0.0,0.0,0.0,0.0,3.0,100.0,8.287822
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
52.52,13.419998,2026-03-21,21:51:07.412808,2026-03-27 19:00:00 UTC,2026-03-27,"""19""",5.0695,64.0,1.824636,14.0,0.0,0.0,0.0,0.0,0.0,1.0,30.0,6.48
52.52,13.419998,2026-03-21,21:51:07.412808,2026-03-27 20:00:00 UTC,2026-03-27,"""20""",3.8695,67.0,0.650032,16.0,0.0,0.0,0.0,0.0,0.0,1.0,33.0,5.804825
52.52,13.419998,2026-03-21,21:51:07.412808,2026-03-27 21:00:00 UTC,2026-03-27,"""21""",2.8695,68.0,-0.408439,17.0,0.0,0.0,0.0,0.0,0.0,1.0,35.0,5.506941


In [ ]:
# Initialise the duckdb database
db_token = 

In [10]:
con = duckdb.connect(f'md:?"motherduck_token={db_token}')

In [11]:
con.sql("SHOW DATABASES").show()

┌───────────────────────┐
│     database_name     │
│        varchar        │
├───────────────────────┤
│ WeatherData           │
│ md_information_schema │
│ my_db                 │
│ sample_data           │
└───────────────────────┘



In [12]:
df= con.sql("SELECT * FROM sample_data.nyc.rideshare LIMIT 10").pl()

In [13]:
df.head()

hvfhs_license_num,dispatching_base_num,originating_base_num,request_datetime,on_scene_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,trip_time,base_passenger_fare,tolls,bcf,sales_tax,congestion_surcharge,airport_fee,tips,driver_pay,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag
str,str,str,datetime[μs],datetime[μs],datetime[μs],datetime[μs],i64,i64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,str
"""HV0003""","""B03404""","""B03404""",2022-11-01 00:04:09,2022-11-01 00:10:11,2022-11-01 00:10:31,2022-11-01 00:17:28,61,61,1.04,417,7.91,0.0,0.24,0.7,0.0,0.0,0.0,5.4,"""N""","""N""",""" ""","""N""","""N"""
"""HV0003""","""B03404""","""B03404""",2022-11-01 00:42:26,2022-11-01 00:45:33,2022-11-01 00:46:33,2022-11-01 00:58:18,209,79,2.07,705,14.48,0.0,0.43,1.29,2.75,0.0,0.0,11.13,"""N""","""N""",""" ""","""N""","""N"""
"""HV0003""","""B03404""","""B03404""",2022-11-01 00:11:53,2022-11-01 00:17:15,2022-11-01 00:19:16,2022-11-01 00:46:19,181,170,8.27,1623,27.37,0.0,0.82,2.43,2.75,0.0,0.0,24.14,"""N""","""N""",""" ""","""N""","""N"""
"""HV0003""","""B03404""","""B03404""",2022-11-01 00:45:30,2022-11-01 00:49:39,2022-11-01 00:50:18,2022-11-01 01:08:10,107,80,5.02,1072,26.74,0.0,0.8,2.37,2.75,0.0,0.0,17.0,"""N""","""N""",""" ""","""N""","""N"""
"""HV0005""","""B03406""",null,2022-10-31 23:56:33,null,2022-11-01 00:01:41,2022-11-01 00:20:33,41,241,6.322,1132,22.34,0.0,0.67,1.98,0.0,0.0,0.0,17.4,"""N""","""N""","""N""","""N""","""N"""


In [14]:
df_raw = df_raw.to_arrow()

In [15]:
con.sql('CREATE TABLE my_db.raw_test AS SELECT * FROM df_raw')

In [ ]:
# send the df_raw into the staging table - label it as such